In [2]:
import re

import pandas as pd

## 1. Read and parse text file

In [3]:
def parse_real_estate_data(raw_data: str) -> pd.DataFrame:
    '''
    Turn real_estate.txt into a dataframe

    Parameters:
        raw_data: content string of real_estate.txt

    Returns:
        pd.DataFrame containing extracted records
    '''

    record_blocks = re.split(r'\n-+\n', raw_data.strip())

    all_records = []
    
    for block in record_blocks:
        if not block.strip():
            continue

        record = {}
        lines = block.strip().split('\n')
        
        for line in lines:
            if ':' in line:
                # Split at the first colon only to preserve full url & desc
                parts = line.split(':', maxsplit=1)

                key = parts[0].strip()
                value = parts[1].strip()
                
                record[key] = value

        all_records.append(record)

    df = pd.DataFrame(all_records)

    return df

In [4]:
with open('../data/raw/real_estate.txt', encoding='utf-8') as file:
    raw = file.read() 

df = parse_real_estate_data(raw)
df.head()

,ID,URL,Title,Price,Area,Bedrooms,Bathrooms,Legal,Interior,Facing Direction,Balcony Direction,Front Width,Front Road Width,Description,Verified,Location,Scraped At,LH
0,1,https://batdongsan.com.vn/ban-can-ho-chung-cu-...,"Vô cùng hối tiếc khi không mua, The Gió Rivers...","2,47 tỷ",65 m²,2 phòng,2 phòng,Hợp đồng mua bán,Đầy đủ,None,None,None,None,"The Gió Riverside của CĐT An Gia, chính thức n...",Yes,"Dự án The Gió Riverside, Đường Vành Đai 3, Phư...",2025-10-05 06:29:49,NaN
1,2,https://batdongsan.com.vn/ban-can-ho-chung-cu-...,The Gió - Thanh toán tối đa chỉ 450tr trong 3 ...,Thỏa thuận,"65,1 m²",2 phòng,2 phòng,Hợp đồng mua bán,Cơ bản,None,None,None,None,Booking sớm chọn giỏ hàng đẹp căn hộ The Gió R...,Yes,"Dự án The Gió Riverside, Đường ĐT 16, Phường B...",2025-10-05 06:29:59,NaN
2,3,https://batdongsan.com.vn/ban-can-ho-chung-cu-...,BQL Vinhomes Smart City cập nhật quỹ căn: Stud...,"4,6 tỷ","64,6 m²",2 phòng,2 phòng,Sổ đỏ/ Sổ hồng.,Đầy đủ.,Tây - Bắc,Đông - Nam,None,None,Em Hoàng Giang là cư dân sống tại Vinhomes Sm...,Yes,"Dự án The Sapphire-Vinhomes Smart City, Phường...",2025-10-05 06:30:08,NaN
3,4,https://batdongsan.com.vn/ban-can-ho-chung-cu-...,"Chỉ cần 200tr, trả góp 4tr5/tháng sở hữu nhà n...","1,89 tỷ","55,3 m²",1 phòng,1 phòng,None,Cơ bản. Nội thất: NT cơ bản đến từ các thương ...,None,Tây - Nam,None,None,Em xin gửi đến anh chị thông tin tổng hợp dự á...,Yes,"Dự án Phú Đông SkyOne, Đường ĐT 743C, Phường T...",2025-10-05 06:30:20,NaN
4,5,https://batdongsan.com.vn/ban-can-ho-chung-cu-...,"Giá thật! CK cao nhất 10/2025, trực tiếp CĐT 1...","3,57 tỷ","52,6 m²",1 phòng,1 phòng,Hợp đồng mua bán.,None,None,None,None,None,"Chính sách mới của CĐT Gamuda Land 10/2025, mở...",Yes,"Dự án Elysian, Đường Lò Lu, Phường Trường Thạn...",2025-10-05 06:30:31,NaN


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2722 entries, 0 to 2721
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   ID                 2722 non-null   object
 1   URL                2722 non-null   object
 2   Title              2722 non-null   object
 3   Price              2722 non-null   object
 4   Area               2722 non-null   object
 5   Bedrooms           2722 non-null   object
 6   Bathrooms          2722 non-null   object
 7   Legal              2722 non-null   object
 8   Interior           2722 non-null   object
 9   Facing Direction   2722 non-null   object
 10  Balcony Direction  2722 non-null   object
 11  Front Width        2722 non-null   object
 12  Front Road Width   2722 non-null   object
 13  Description        2722 non-null   object
 14  Verified           2722 non-null   object
 15  Location           2722 non-null   object
 16  Scraped At         2722 non-null   object


## 2. Basic cleanup

### &emsp; 2.1. Extract values

#### &emsp; &emsp; 2.1.1. Extract numeric values

In [6]:
def extract_numeric(s:str|None) -> float:
    if not isinstance(s,str) or not s:
        return None

    results = re.search(r"^(\d+.?\d*,?\d*)\D?", s)
    if not results:
        return None
    else:
        num_str = results.group(1)
        num_str = num_str.replace(".","")
        num_str = num_str.replace(",",".")
        return float(num_str)
    
def extract_measuring_unit(s:str|None) -> str:
    if not isinstance(s,str) or not s:
        return None

    results = re.search(r"^\d+.?\d*,?\d*\s*(\D*)", s)
    if not results:
        return None
    else:
        return results.group(1)

In [7]:
df["price_val"] = df["Price"].apply(extract_numeric)
df["price_unit"] = df["Price"].apply(extract_measuring_unit)
df["area_val"] = df["Area"].apply(extract_numeric)
df["area_unit"] = df["Area"].apply(extract_measuring_unit)
df["n_bedrooms"] = df["Bedrooms"].apply(extract_numeric)
df["n_bathrooms"] = df["Bathrooms"].apply(extract_numeric)
df["front_width_val"] = df["Front Width"].apply(extract_numeric)
df["front_width_unit"] = df["Front Width"].apply(extract_measuring_unit)
df["front_road_width_val"] = df["Front Road Width"].apply(extract_numeric)
df["front_road_width_unit"] = df["Front Road Width"].apply(extract_measuring_unit)

df[[
    "price_val", "price_unit", "area_val", "area_unit", "n_bedrooms", "n_bathrooms", 
    "front_width_val", "front_width_unit", "front_road_width_val", "front_road_width_unit"
]].describe(include="all")

,price_val,price_unit,area_val,area_unit,n_bedrooms,n_bathrooms,front_width_val,front_width_unit,front_road_width_val,front_road_width_unit
count,2383.00000,2383,2722.000000,2722,1748.000000,1652.000000,1133.000000,1133,992.000000,992
unique,NaN,5,NaN,1,NaN,NaN,NaN,1,NaN,1
top,NaN,tỷ,NaN,m²,NaN,NaN,NaN,m,NaN,m
freq,NaN,2084,NaN,2722,NaN,NaN,NaN,1133,NaN,992
mean,35.01859,NaN,329.568916,NaN,3.486842,3.234867,9.211192,NaN,13.690524,NaN
std,102.14035,NaN,2546.525614,NaN,4.580058,4.732382,13.567192,NaN,12.586475,NaN
min,1.00000,NaN,13.000000,NaN,1.000000,1.000000,1.000000,NaN,1.000000,NaN
25%,4.20000,NaN,62.500000,NaN,2.000000,2.000000,5.000000,NaN,6.000000,NaN
50%,9.40000,NaN,88.000000,NaN,3.000000,2.000000,6.000000,NaN,11.500000,NaN
75%,23.45000,NaN,140.000000,NaN,4.000000,4.000000,10.000000,NaN,17.000000,NaN


#### &emsp;&emsp; 2.1.2. Handling inconsistent units

In [8]:
df["price_unit"].unique()

array(['tỷ', None, 'triệu/m²', 'triệu', 'tỷ/m²', 'nghìn'], dtype=object)

Checking suspicious prices (with "nghìn" unit)

In [9]:
df.loc[(df.price_unit == 'nghìn')]

,ID,URL,Title,Price,Area,Bedrooms,Bathrooms,Legal,Interior,Facing Direction,...,price_val,price_unit,area_val,area_unit,n_bedrooms,n_bathrooms,front_width_val,front_width_unit,front_road_width_val,front_road_width_unit
2288,2289,https://batdongsan.com.vn/ban-nha-rieng-duong-...,"Bán gấp NR 3PN, 2WC, 80m2, 3tỷ 370tại Mai Đăng...",3 nghìn,80 m²,3 phòng,2 phòng,Sổ đỏ/ Sổ hồng,Cơ bản,Nam,...,3.0,nghìn,80.0,m²,3.0,2.0,7.45,m,4.0,m
2402,2403,https://batdongsan.com.vn/ban-dat-duong-tinh-l...,Hơn 200m2 full thổ cư tại triệu thành - Triệu ...,300 nghìn,200 m²,None,None,Sổ đỏ/ Sổ hồng,None,Đông - Bắc,...,300.0,nghìn,200.0,m²,NaN,NaN,5.00,m,5.0,m


Converting all prices to million (or million/m^2)

In [10]:
CONVERSION_MAP = {
    "tỷ": 1000,
    "tỷ/m²": 1000,
    "triệu": 1,
    "triệu/m²": 1,
}
def price_to_mil(row, val_col_label:str, unit_col_label:str):
    value = row[val_col_label]
    unit = row[unit_col_label]
    conversion_factor = CONVERSION_MAP.get(unit)
    if value and unit:
        return value * conversion_factor
    else:
        return None

In [11]:
# correct the wrong price of records with "nghìn" price_unit based on their titles
df.loc[2288, 'price_val'] = 3370
df.loc[2402, 'price_val'] = 299
df.loc[[2288, 2402], 'price_unit'] = 'triệu'

# convert all to million
df["price_val"] = df.apply(price_to_mil, axis=1, args=("price_val","price_unit"))

# convert price per m2 units to full price
price_per_area = df.price_unit.isin(["tỷ/m²", "triệu/m²"])
df.loc[price_per_area, "price_val"] = df.loc[price_per_area, "price_val"] * df.loc[price_per_area, "area_val"]

df[[
    "price_val", "area_val", "n_bedrooms", "n_bathrooms", 
    "front_width_val", "front_road_width_val",
]].describe()

,price_val,area_val,n_bedrooms,n_bathrooms,front_width_val,front_road_width_val
count,2.383000e+03,2722.000000,1748.000000,1652.000000,1133.000000,992.000000
mean,1.654823e+04,329.568916,3.486842,3.234867,9.211192,13.690524
std,3.860134e+04,2546.525614,4.580058,4.732382,13.567192,12.586475
min,1.090000e+01,13.000000,1.000000,1.000000,1.000000,1.000000
25%,3.700000e+03,62.500000,2.000000,2.000000,5.000000,6.000000
50%,7.665000e+03,88.000000,3.000000,2.000000,6.000000,11.500000
75%,1.700000e+04,140.000000,4.000000,4.000000,10.000000,17.000000
max,1.200000e+06,75000.000000,100.000000,110.000000,355.000000,200.000000


#### &emsp;&emsp; 2.1.3. Extract `property_type` from `URL`

In [12]:
PROPERTY_LINKS_TYPES = {
    "ban-can-ho-chung-cu-mini":"Căn hộ chung cư mini",
    "ban-can-ho-chung-cu": "Căn hộ chung cư",
    "ban-nha-rieng": "Nhà riêng",
    "ban-nha-biet-thu-lien-ke": "Nhà biệt thự, liền kề",
    "ban-nha-mat-pho": "Nhà mặt phố",
    "ban-shophouse-nha-pho-thuong-mai": "Shophouse, nhà phố thương mại",
    "ban-dat-nen-du-an": "Đất nền dự án",
    "ban-dat": "Đất",
    "ban-condotel": "Condotel",
    "ban-trang-trai-khu-nghi-duong": "Trang trại, khu nghỉ dưỡng",
    "ban-kho-nha-xuong": "Kho, nhà xưởng",
    "ban-loai-bat-dong-san-khac": "Khác"
}

def extract_propterty_type(url:str) -> str:
    results = re.search(r'^https://batdongsan.com.vn/(.*)/', url)

    if not results:
        return None
    
    type_n_place = results.group(1)

    for type_link_affix in PROPERTY_LINKS_TYPES:
        if type_link_affix in type_n_place:
            return PROPERTY_LINKS_TYPES[type_link_affix]
        
    return None

def shorten_url(url:str) -> str:
    return url[26:]

In [13]:
df['property_type'] = df.URL.apply(extract_propterty_type)
df['short_url'] = df.URL.apply(shorten_url)
df[['property_type', 'short_url']].sample(5)

,property_type,short_url
2307,Đất,ban-dat-duong-dt-750-xa-long-hoa-1/-nen-mat-ti...
614,Đất nền dự án,ban-dat-nen-du-an-duong-dt-769-xa-binh-son-4-p...
489,Căn hộ chung cư,ban-can-ho-chung-cu-duong-29-3-1-phuong-hoa-xu...
1391,"Nhà biệt thự, liền kề",ban-nha-biet-thu-lien-ke-xa-long-hung-6-prj-aq...
2118,Condotel,ban-condotel-duong-tran-phu-phuong-5-18-prj-me...


#### &emsp;&emsp; 2.1.4. Extract `city` from `Location`

In [24]:
def extract_location_detail(addr: str, level:int) -> str|None:
    '''
    level:int - the level of details to extract from the address. 1 -> Province; 2 -> District
    '''
    if not isinstance(addr, str) or not addr.strip():
        return None
    parts = [p.strip() for p in addr.split(",") if p.strip() != ""]
    if len(parts) >= level:
        result = parts[-level].replace('.','')
        return result
    return None

In [25]:
df['city_province'] = df.Location.apply(extract_location_detail, level = 1)
df['district'] = df.Location.apply(extract_location_detail, level = 2)
df[['Location', 'city_province', 'district']].sample(5)

,Location,city_province,district
410,"Ruby Park, Phúc Lợi, Phúc Lợi, Long Biên, Hà Nội",Hà Nội,Long Biên
2343,"Phường Mỹ Đình 1, Nam Từ Liêm, Hà Nội",Hà Nội,Nam Từ Liêm
2662,"Dự án An Khang Villa, Đường Tố Hữu, Phường La ...",Hà Nội,Hà Đông
1338,"Dự án An Bình Homeland, Phường Dương Nội, Hà Đ...",Hà Nội,Hà Đông
1427,"Đường Điện Biên Phủ, Phường 25, Bình Thạnh, Hồ...",Hồ Chí Minh,Bình Thạnh


Fix null cities & Standardize city labels

In [26]:
df.city_province.unique()

array(['Bình Dương', 'Hà Nội', 'Hồ Chí Minh', 'Đà Nẵng', 'Quảng Nam',
       'Hưng Yên', '86Tỷ', 'Bắc Giang', 'Khánh Hòa', 'Quảng Ninh',
       'Vĩnh Phúc', 'Thành phố Hà Nội', 'Long An', 'Hải Phòng',
       'Lâm Đồng', 'Bà Rịa Vũng Tàu', 'Nghệ An', 'Bắc Ninh', 'Nam Định',
       'Quảng Ngãi', 'Đồng Nai', 'Hà Nam', 'Thừa Thiên Huế', 'Thanh Hóa',
       'Bình Phước', 'Thành phố Hồ Chí Minh', 'Hòa Bình', 'Thái Nguyên',
       'Tây Ninh', 'Bạc Liêu', 'Hậu Giang', 'Bình Định', 'Hải Dương',
       'Cần Thơ', 'Đường Nguyễn Cửu Phú', 'Bình Thuận', 'Bến Tre',
       'Thái Bình', 'Phú Thọ', 'Kiên Giang', 'Ninh Bình',
       'Chung cư mipec city view kiến hưng hà đông hà nội', 'Ninh Thuận',
       'thành phố Hồ Chí Minh', 'Tiền Giang', 'Quảng Trị',
       'Tân quang văn lâm hưng yên', 'Quảng Bình'], dtype=object)

In [34]:
df.loc[[1684, 2192, 2561], 'city_province'] = ['TP.HCM', 'Hà Nội', 'Hưng Yên']
df.loc[df.city_province == '86Tỷ', 'city_province'] = 'TP.HCM'
df.loc[df.city_province.isin(['Thành phố Hồ Chí Minh', 'thành phố Hồ Chí Minh', 'Hồ Chí Minh', 'TPHCM']), 'city_province'] = 'TP.HCM'
df.loc[df.city_province == 'Thành phố Hà Nội', 'city_province'] = 'TP.HCM'

In [35]:
df.city_province.unique()

array(['Bình Dương', 'Hà Nội', 'TP.HCM', 'Đà Nẵng', 'Quảng Nam',
       'Hưng Yên', 'Bắc Giang', 'Khánh Hòa', 'Quảng Ninh', 'Vĩnh Phúc',
       'Long An', 'Hải Phòng', 'Lâm Đồng', 'Bà Rịa Vũng Tàu', 'Nghệ An',
       'Bắc Ninh', 'Nam Định', 'Quảng Ngãi', 'Đồng Nai', 'Hà Nam',
       'Thừa Thiên Huế', 'Thanh Hóa', 'Bình Phước', 'Hòa Bình',
       'Thái Nguyên', 'Tây Ninh', 'Bạc Liêu', 'Hậu Giang', 'Bình Định',
       'Hải Dương', 'Cần Thơ', 'Bình Thuận', 'Bến Tre', 'Thái Bình',
       'Phú Thọ', 'Kiên Giang', 'Ninh Bình', 'Ninh Thuận', 'Tiền Giang',
       'Quảng Trị', 'Quảng Bình'], dtype=object)

#### &emsp;&emsp; 2.1.5. Convert "None" strings to `None`

In [36]:
df.replace("None", None, inplace=True)

#### &emsp;&emsp; 2.1.6. Standardize legal document labels

In [37]:
def standardize_legal_doc_types(s:str) -> str:
    if not s:
        return None
    
    s = s.lower().strip()
    if re.search(r'/', s) or re.search(r'có sổ', s) or re.search(r'có số', s) or re.search(r'đã có sổ', s) or re.search(r'đã có sổ đỏ', s) or re.search(r'đã có sổ hồng', s) or re.search(r'có sổ hồng', s):
        return 'Có sổ'
    if re.search(r'sổ đỏ', s) or re.search(r'sđ', s):
        return 'Sổ đỏ'
    if re.search(r'sổ hồng', s):
        return 'Sổ hồng'
    if re.search(r'hợp đồng mua bán', s) or re.search(r'hđmb', s) or re.search(r'hdmb', s) or re.search(r'hợp đồng góp vốn', s):
        return 'Hợp đồng mua bán'
    if re.search(r'đầy đủ', s) or re.search(r'chính chủ', s) or re.search(r'lâu dài', s) or re.search(r'giấy phép', s) or re.search(r'giấy tờ', s) or re.search(r'vĩnh viễn', s):
        return 'Khác'
    return 'Khác'

In [38]:
df['legal_docs'] = df.Legal.apply(standardize_legal_doc_types)
df[['legal_docs', 'Legal']].sample(5)

,legal_docs,Legal
839,None,None
28,Có sổ,Sổ đỏ/ Sổ hồng
1109,Có sổ,Sổ đỏ/ Sổ hồng
591,Có sổ,Sổ đỏ/ Sổ hồng
2324,Sổ đỏ,Sổ đỏ


### &emsp; 2.2. Select columns for merging and analysis

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2722 entries, 0 to 2721
Data columns (total 34 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ID                     2722 non-null   object 
 1   URL                    2722 non-null   object 
 2   Title                  2722 non-null   object 
 3   Price                  2722 non-null   object 
 4   Area                   2722 non-null   object 
 5   Bedrooms               1748 non-null   object 
 6   Bathrooms              1652 non-null   object 
 7   Legal                  2327 non-null   object 
 8   Interior               1431 non-null   object 
 9   Facing Direction       1177 non-null   object 
 10  Balcony Direction      822 non-null    object 
 11  Front Width            1133 non-null   object 
 12  Front Road Width       992 non-null    object 
 13  Description            2721 non-null   object 
 14  Verified               2722 non-null   object 
 15  Loca

In [43]:
cols_for_analysis = [
    'price_val', 'area_val', 'n_bedrooms', 'n_bathrooms', 'front_width_val', 'front_road_width_val',
    'legal_docs', 'Facing Direction', 'Balcony Direction', 'property_type', 'city_province', 'district', 'Location',
    # misc
    'URL', 'Title', 'Description', 'Interior'
]
df_pruned = df[cols_for_analysis]
df_pruned.sample(5)

,price_val,area_val,n_bedrooms,n_bathrooms,front_width_val,front_road_width_val,legal_docs,Facing Direction,Balcony Direction,property_type,city_province,district,Location,URL,Title,Description,Interior
2413,68000.0,480.0,NaN,NaN,16.0,20.0,Hợp đồng mua bán,Nam,Nam,"Nhà biệt thự, liền kề",Bắc Ninh,Từ Sơn,"Dự án Centa Riverside, Đường Hữu Nghị, Phường ...",https://batdongsan.com.vn/ban-nha-biet-thu-lie...,"Mở bán biệt thự đơn lập view sông 480m2, giá g...",Mở bán biệt thự đơn lập view sông 480m² giá gố...,Không nội thất
1648,1970.0,55.0,2.0,1.0,NaN,NaN,Có sổ,Bắc,Tây,Căn hộ chung cư,Bình Dương,Dĩ An,"Dự án Charm City, Đường ĐT 743, Phường Dĩ An, ...",https://batdongsan.com.vn/ban-can-ho-chung-cu-...,"Cần bán căn hộ Charm City 1PN 50m2 1tỷ750, 2PN...",- Cần bán căn hộ chung cư vào ở ngay sẵn sổ hồ...,Cơ bản.
1671,15000.0,225.0,NaN,NaN,NaN,NaN,None,None,None,Đất,Đà Nẵng,Cẩm Lệ,"Đường Nguyễn Phước Lan, Phường Hòa Xuân, Cẩm L...",https://batdongsan.com.vn/ban-dat-duong-nguyen...,Sun Group mở bán đất mặt tiền trục Nguyễn Phướ...,"Sun Group sắp mở bán hơn 1000 lô đất nền, biệt...",None
1938,12500.0,100.0,NaN,NaN,4.5,10.0,Có sổ,Bắc,None,Đất,Hà Nội,Thường Tín,"Xã Nhị Khê, Thường Tín, Hà Nội",https://batdongsan.com.vn/ban-dat-xa-nhi-khe/b...,Đầu tư sinh lời: Đất Nhị Khê chỉ 125 triệu/m² ...,"Nhanh tay sở hữu đất phân lô tại Nhị Khê, Thườ...",None
2590,NaN,806.0,NaN,NaN,NaN,NaN,Có sổ,None,None,Đất,Hòa Bình,Lương Sơn,"Đường Đồng Chanh, Xã Nhuận Trạch, Lương Sơn, H...",https://batdongsan.com.vn/ban-dat-duong-dong-c...,Bán 806 m2 đất bám hồ Đồng Chanh Lương Sơn Hoà...,Diện tích 806 m² có 320 m² thổ cư còn lại đất ...,None


## 3. Check and export

In [44]:
rename_map = {
    'price_val': 'price',
    'area_val': 'area',
    'front_width_val': 'front_width', 
    'front_road_width_val': 'front_road_width', 
    'Facing Direction': 'facing_direction',
    'Balcony Direction': 'balcony_direction',
    'Location': 'address',
    'URL': 'url',
    'Title': 'title',
    'Description': 'description',
    'Interior': 'interior'
}
df_pruned = df_pruned.rename(rename_map, axis=1)

In [45]:
df_pruned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2722 entries, 0 to 2721
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   price              2383 non-null   float64
 1   area               2722 non-null   float64
 2   n_bedrooms         1748 non-null   float64
 3   n_bathrooms        1652 non-null   float64
 4   front_width        1133 non-null   float64
 5   front_road_width   992 non-null    float64
 6   legal_docs         2327 non-null   object 
 7   facing_direction   1177 non-null   object 
 8   balcony_direction  822 non-null    object 
 9   property_type      2722 non-null   object 
 10  city_province      2722 non-null   object 
 11  district           2719 non-null   object 
 12  address            2722 non-null   object 
 13  url                2722 non-null   object 
 14  title              2722 non-null   object 
 15  description        2721 non-null   object 
 16  interior           1431 

In [54]:
df_pruned.city_province.unique()

array(['Bình Dương', 'Hà Nội', 'TP.HCM', 'Đà Nẵng', 'Quảng Nam',
       'Hưng Yên', 'Bắc Giang', 'Khánh Hòa', 'Quảng Ninh', 'Vĩnh Phúc',
       'Long An', 'Hải Phòng', 'Lâm Đồng', 'Bà Rịa Vũng Tàu', 'Nghệ An',
       'Bắc Ninh', 'Nam Định', 'Quảng Ngãi', 'Đồng Nai', 'Hà Nam',
       'Thừa Thiên Huế', 'Thanh Hóa', 'Bình Phước', 'Hòa Bình',
       'Thái Nguyên', 'Tây Ninh', 'Bạc Liêu', 'Hậu Giang', 'Bình Định',
       'Hải Dương', 'Cần Thơ', 'Bình Thuận', 'Bến Tre', 'Thái Bình',
       'Phú Thọ', 'Kiên Giang', 'Ninh Bình', 'Ninh Thuận', 'Tiền Giang',
       'Quảng Trị', 'Quảng Bình'], dtype=object)

In [53]:
df_pruned.to_csv('../data/interim/batdongsan_com_vn(2).csv')